## imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import sklearn

sys.path.append('../src')
from load_clean_data import load_clean_data
from helpers import evaluate_forecast

In [2]:
df = load_clean_data()
print(df.shape)

c:\Users\corbi\Documents\Projects\sales-forecasting\sales-forecasting-project\notebooks\../src\load_clean_data.py:6: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('../data/train.csv')


(1017209, 18)


### Split Dataset

In [3]:
train = df[df['Date'] < '2015-06-30']
test  = df[df['Date'] >= '2015-06-30']

length_check = len(train) + len(test) == len(df)

print(f"Length check: {length_check}")

Length check: True


#### Store 1

In [4]:
store_1_train = train[train['Store'] == 1]
store_1_test  = test[test['Store'] == 1]

print(f"Store 1 train shape: {store_1_train.shape}")
print(f"Store 1 test shape: {store_1_test.shape}")

Store 1 train shape: (910, 18)
Store 1 test shape: (32, 18)


### Seasonal Average

#### Full History Average

In [10]:
seasonal_average_forecast = store_1_train.groupby(['Store', 'DayOfWeek'])['Sales'].mean()
print(seasonal_average_forecast)

seasonal_average_forecast = seasonal_average_forecast.reset_index()

Store  DayOfWeek
1      1            4945.153846
       2            4652.907692
       3            4456.315385
       4            4077.700000
       5            4518.630769
       6            4963.546154
       7               0.000000
Name: Sales, dtype: float64


In [12]:
seasonal_average_forecast = seasonal_average_forecast.rename(columns={'Sales': 'SeasonalAverage'})

store_1_test_w_seasonal_avg = store_1_test.merge(seasonal_average_forecast, on=['Store', 'DayOfWeek'], how='left')

print(store_1_test_w_seasonal_avg.shape)
print()
print(store_1_test_w_seasonal_avg.isnull().sum())
print()
print(store_1_test_w_seasonal_avg[['Store', 'DayOfWeek', 'Date', 'Sales', 'SeasonalAverage']].sort_values('Date').head(10))

(32, 19)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
SeasonalAverage              0
dtype: int64

    Store  DayOfWeek       Date  Sales  SeasonalAverage
31      1          2 2015-06-30   5735      4652.907692
30      1          3 2015-07-01   5223      4456.315385
29      1          4 2015-07-02   5558      4077.700000
28      1          5 2015-07-03   4665      4518.630769
27      1          6 2015-07-04   4797      4963.546154
26      1          7 2015-07-05      0         0.00